In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

###dim_date

In [0]:
from pyspark.sql.functions import col, date_format, expr, weekofyear, dayofweek, lit
from pyspark.sql.types import IntegerType

start_date = "2026-04-01"
end_date = "2026-04-30"

dim_date = (
    spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as d_array")
    .select(expr("explode(d_array)").alias("calendar_date"))
    .select(
        date_format(col("calendar_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        col("calendar_date").cast("date").alias("calendar_date"),
        date_format(col("calendar_date"), "EEEE").alias("day_name"),
        weekofyear(col("calendar_date")).cast(IntegerType()).alias("iso_week"),
        (dayofweek(col("calendar_date")).isin(1, 7)).alias("is_weekend"),
    )
)

(dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_date"))

###dim_station

In [0]:
from pyspark.sql.functions import *

dim_store = (
    spark.read.table(f"{catalog}.silver.stations")
    .select(
        col("station_id"),
        col("station_name"),
        col("city"),
        col("site_type"),
        col("opened_on"),
    )
)

(dim_store.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_station"))

###dim_charger

In [0]:
from pyspark.sql.functions import *

dim_products = (
    spark.read.table(f"{catalog}.silver.chargers")
    .join(
        spark.read.table(f"{catalog}.silver.stations").select(col("station_id"), col("city")),
        on="station_id",
        how="left",
    )
    .select(
        col("charger_id"),
        col("commissioned_on"),
        col("connector"),
        col("is_active"),
        col("snapshot_date"),
        col("station_id"),
        col("city"),
        col("connector_type"),
        col("max_kw"),
    )
    .dropDuplicates(["charger_id"])
)

(dim_products.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_charger"))

###dim_member

In [0]:
from pyspark.sql.functions import *

FROM_COL = "effective_from"
TO_COL = "effective_to"

dim_customers = (
    spark.read.table(f"{catalog}.silver.members")
    .withColumn("member_sk", monotonically_increasing_id())
    .select(
        col("member_sk").cast("bigint"),
        col("member_id").cast("string"),
        col("full_name"),
        col("email"),
        col("phone"),
        col("city"),
        col("plan"),
        col("joined_on"),
        col(FROM_COL).alias("effective_from"),
        col(TO_COL).alias("effective_to"),
        col("is_current")
    )
)

(
    dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.dim_members")
)

###fact_sessions

In [0]:
from pyspark.sql.functions import *

sessions = spark.read.table(f"{catalog}.silver.sessions")
dim_members = spark.read.table(f"{catalog}.gold.dim_members")
chargers = spark.read.table(f"{catalog}.silver.chargers").dropDuplicates(["charger_id"])
tariffs = spark.read.table(f"{catalog}.silver.tariffs")
faults = spark.read.table(f"{catalog}.silver.session_faults")

fault_counts = faults.groupBy("session_id").agg(count(lit(1)).alias("fault_count"))

fact_sessions = (
    sessions
    .join(chargers.select("charger_id", "connector_type"), on="charger_id", how="left")
    .join(tariffs, on=["connector_type", "time_band"], how="left")
    .join(
        dim_members.select(
            col("member_id"),
            col("member_sk"),
            col("plan").alias("plan_at_session"),
            col("effective_from"),
            col("effective_to"),
        ),
        on="member_id",
        how="left",
    )
    .filter(
        col("effective_from").isNull()
        | (
            (col("effective_from") <= col("session_date"))
            & (col("effective_to").isNull() | (col("session_date") < col("effective_to")))
        )
    )
    .join(fault_counts, on="session_id", how="left")
    .withColumn("date_key", date_format(col("session_date"), "yyyyMMdd").cast("int"))
    .withColumn("plan_at_session", when(col("plan_at_session").isNull(), lit("GUEST")).otherwise(col("plan_at_session")))
    .withColumn("energy_charge", round(col("energy_kwh") * col("rate_per_kwh")))
    .withColumn(
        "plan_discount",
        round(
            col("energy_charge")
            * when(col("plan_at_session") == "PRO", lit(0.10))
              .when(col("plan_at_session") == "FLEET", lit(0.15))
              .otherwise(lit(0.0))
        ),
    )
    .withColumn(
        "idle_fee",
        greatest(col("idle_minutes") - col("free_idle_minutes"), lit(0))
        * col("idle_fee_per_min"),
    )
    .withColumn(
        "billed_amount",
        when(col("status") == "COMPLETED", col("energy_charge") - col("plan_discount") + col("idle_fee"))
        .otherwise(lit(0.00)),
    )
    .withColumn(
        "billing_status",
        when(col("status") == "COMPLETED",
            when(abs(col("amount_paid") - col("billed_amount")) < 0.50, lit("OK"))
            .otherwise(lit("MISMATCH"))
        ),
    )
    .select(
        col("session_id"),
        col("date_key"),
        col("member_id"),
        col("member_sk"),
        col("plan_at_session"),
        col("amount_paid"),
        col("energy_kwh"),
        col("duration_minutes"),
        col("time_band"),
        col("charger_id"),
        col("status"),
        col("idle_minutes"),
        col("rate_per_kwh"),
        col("energy_charge"),
        col("plan_discount"),
        col("idle_fee"),
        col("billed_amount"),
        col("billing_status"),
        col("fault_count"),
    )
)

(fact_sessions.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.fact_sessions"))

###agg_daily_station

In [0]:
from pyspark.sql.functions import *

fact_sessions = spark.read.table(f"{catalog}.gold.fact_sessions")
dim_date      = spark.read.table(f"{catalog}.gold.dim_date")
dim_station   = spark.read.table(f"{catalog}.gold.dim_station")
chargers      = spark.read.table(f"{catalog}.silver.chargers").dropDuplicates(["charger_id"])

fs = (
    fact_sessions
    .join(chargers.select("charger_id", "station_id"), on="charger_id", how="left")
    .join(dim_date.select("date_key", col("calendar_date").alias("session_date")), on="date_key", how="left")
    .join(dim_station.select("station_id", "city"), on="station_id", how="left")
)

agg = (
    fs.groupBy("session_date", "station_id", "city")
    .agg(
        sum(when(col("status") == "COMPLETED", 1).otherwise(0)).alias("completed_sessions"),
        sum(when(col("status") != "COMPLETED", 1).otherwise(0)).alias("failed_sessions"),
        sum(when(col("status") == "COMPLETED", col("energy_kwh")).otherwise(lit(0.0))).alias("energy_kwh"),
        sum(when(col("status") == "COMPLETED", col("billed_amount")).otherwise(lit(0.0))).alias("billed_amount"),
        sum(when(col("status") == "COMPLETED", col("amount_paid")).otherwise(lit(0.0))).alias("amount_paid"),
        sum(when(col("billing_status") == "MISMATCH", 1).otherwise(0)).alias("mismatch_sessions"),
        sum(col("fault_count")).alias("fault_count"),
        sum(when(col("status") == "COMPLETED", col("duration_minutes")).otherwise(lit(0.0))).alias("completed_duration"),
    )
)

bounds = fs.select(min("session_date").alias("min_date"), max("session_date").alias("max_date")).collect()[0]
all_dates = (
    spark.sql(f"SELECT sequence(to_date('{bounds['min_date']}'), to_date('{bounds['max_date']}'), interval 1 day) as d_array")
    .select(expr("explode(d_array)").alias("session_date"))
)

charger_count = chargers.groupBy("station_id").agg(countDistinct("charger_id").alias("charger_count"))

skeleton = dim_station.select("station_id", "city").crossJoin(all_dates)

agg_daily_station = (
    skeleton
    .join(agg, on=["session_date", "station_id", "city"], how="left")
    .join(charger_count, on="station_id", how="left")
    .fillna(0, subset=["completed_sessions", "failed_sessions", "energy_kwh", "billed_amount", "amount_paid", "mismatch_sessions", "fault_count", "completed_duration"])
    .withColumn("utilisation_pct", least(lit(100.0), round(100.0 * col("completed_duration") / (col("charger_count") * 1440), 2)).cast("decimal(5,2)"))
    .select(
        col("session_date"),
        col("station_id"),
        col("city"),
        col("completed_sessions"),
        col("failed_sessions"),
        col("energy_kwh"),
        col("billed_amount"),
        col("amount_paid"),
        col("mismatch_sessions"),
        col("fault_count"),
        col("utilisation_pct"),
    )
)

(agg_daily_station.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.agg_daily_station"))

###v_kpi_summary

In [0]:
from pyspark.sql.functions import *

fact_sessions = spark.read.table(f"{catalog}.gold.fact_sessions")

kpi_summary = (
    fact_sessions
    .agg(
        sum(when(col("status") == "COMPLETED", 1).otherwise(0)).alias("completed_sessions"),
        sum(when(col("status") != "COMPLETED", 1).otherwise(0)).alias("failed_sessions"),
        sum(when(col("status") == "COMPLETED", col("energy_kwh")).otherwise(lit(0.0))).alias("total_energy_kwh"),
        sum(when(col("status") == "COMPLETED", col("billed_amount")).otherwise(lit(0.0))).alias("total_billed"),
        sum(when(col("status") == "COMPLETED", col("amount_paid")).otherwise(lit(0.0))).alias("total_paid"),
        sum(when(col("billing_status") == "MISMATCH", 1).otherwise(0)).alias("mismatch_sessions"),
        countDistinct(when(col("status") == "COMPLETED", col("member_id"))).alias("active_members"),
    )
    .withColumn(
        "success_rate_pct",
        round(
            col("completed_sessions") / (col("completed_sessions") + col("failed_sessions")) * 100,
            2,
        ).cast("decimal(5,2)"),
    )
    .withColumn(
        "revenue_leakage",
        round(col("total_billed") - col("total_paid"), 2),
    )
    .select(
        col("completed_sessions"),
        col("failed_sessions"),
        col("success_rate_pct"),
        col("total_energy_kwh"),
        col("total_billed"),
        col("total_paid"),
        col("revenue_leakage"),
        col("mismatch_sessions"),
        col("active_members"),
    )
)

(kpi_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.v_kpi_summary"))

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.gold.dim_date IS 'This table contains all dates between 2026-04-01 and 2026-04-30';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.gold.dim_station IS 'This table contains dimesions of charging stations';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.gold.dim_charger IS 'This table contains dimesions of charging connectors';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.gold.dim_members IS 'This table contains dimesions of members';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.gold.fact_sessions IS 'This table contains all sessions';
""");